In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Per-iterate fixed-point metrics. This ships as `fp_metrics.npz`, produced by replaying the
# saved optimisation checkpoints with `track_fp.py` (the offline path). The in-loop path
# (Force_Minimizer with `track_islands = true`) records the same arrays into the run's HDF5
# `/iterations/fp` group. See README.md to regenerate either way.
NPZ_FILE = "fp_metrics.npz"

# Tracking a magnetic island across a SPECTRE optimisation

`spectre.fixed_points.FixedPointTracker` follows the magnetic axis and one island fixed-point
chain as an equilibrium is optimised. This example tracks the **3/5 island chain** in volume 3 of a
fixed-boundary **NCSX** equilibrium (`Nfp = 3`, so `iota = 3/5` with `poloidal_occurrences = 3`,
`repeat_period = 5`) across 21 force-minimisation iterates. The equilibrium is generated from the
public `wout_li383_low_res_reference.nc` VMEC output (see `README.md`).

The 3/5 resonant surface starts just inside the neighbouring volume, so the tracker reports `no_fp`
at iterate 0 and then discovers the chain in volume 3 from iterate 1 on, as the equilibrium relaxes
and the surface moves into the tracked volume. Below we plot how the magnetic axis and the island
O-point move, and how the island's Greene residue evolves. The metrics were produced with the
offline `track_fp.py` path; the in-loop path (`Force_Minimizer` with `track_islands = true`) records
the same quantities into the run's HDF5 `/iterations/fp` group.

In [2]:
d = np.load(NPZ_FILE)
iters = d["iteration_indices"].astype(int)
axis_rz = d["axis_rz"]
island_rz = d["island_rz"]  # tracked O-point; NaN on no_fp iterates
island_rz_hp = d["island_rz_half_period"]  # point half a chain away from the O-point
residue = d["residue"]  # Greene residue R = (2 - tr M) / 4
iota = d["iota"]
iota_rel = d["iota_rel"]
no_fp = d["no_fp"]

found = ~no_fp  # iterates where the island fixed point was located
imin, imax = np.nanmin(iota[found]), np.nanmax(iota[found])
rmin, rmax = np.nanmin(residue[found]), np.nanmax(residue[found])
print(
    f"{len(iters)} iterates; island located in {found.sum()} of them "
    f"(iter {iters[found][0]} .. {iters[found][-1]}), no_fp in {no_fp.sum()}."
)
print(f"iota (located): {imin:.6f} .. {imax:.6f}  (3/5 = {3 / 5:.6f})")
print(f"Greene residue (located): {rmin:.2e} .. {rmax:.2e}  (0 < R < 1 => elliptic O-point)")

21 iterates; island located in 20 of them (iter 1 .. 20), no_fp in 1.
iota (located): 0.600000 .. 0.600000  (3/5 = 0.600000)
Greene residue (located): 3.95e-02 .. 1.91e-01  (0 < R < 1 => elliptic O-point)


## 1 — Magnetic axis position over optimisation iterates

The magnetic axis is tracked as a period-1 fixed point. It drifts as the equilibrium is optimised,
while `Z` stays near 0 (stellarator symmetry).

In [3]:
axis_found = ~np.isnan(axis_rz[:, 0])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(iters[axis_found], axis_rz[axis_found, 0], "o-", ms=3, lw=1)
axes[0].set(xlabel="Optimisation iterate", ylabel="R [m]", title="Magnetic axis R")

axes[1].plot(iters[axis_found], axis_rz[axis_found, 1], "o-", ms=3, lw=1, color="tab:orange")
axes[1].set(xlabel="Optimisation iterate", ylabel="Z [m]", title="Magnetic axis Z")

fig.tight_layout()
plt.show()

## 2 — Island O-point position and Greene residue over iterates

`island_rz` is the tracked O-point (island centre); `island_rz_half_period` is the point half a
chain away from it. Only the iterates where the chain was located in volume 3 are plotted (iterate 0
is `no_fp`). The **Greene residue** `R = (2 − tr M)/4` classifies the point: `0 < R < 1` is elliptic
(an O-point), `R < 0` or `R > 1` is hyperbolic (an X-point). The chain stays elliptic once found.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for col, (label, rz) in enumerate([("O-point", island_rz), ("half-period", island_rz_hp)]):
    axes[0, col].plot(iters[found], rz[found, 0], "o-", ms=3, lw=1)
    axes[0, col].set(xlabel="Optimisation iterate", ylabel="R [m]", title=f"Island {label} R")
    axes[1, col].plot(iters[found], rz[found, 1], "o-", ms=3, lw=1, color="tab:orange")
    axes[1, col].set(xlabel="Optimisation iterate", ylabel="Z [m]", title=f"Island {label} Z")

fig.tight_layout()
plt.show()

# Greene residue R = (2 - tr M) / 4: 0 < R < 1 is elliptic (O-point); outside that is hyperbolic.
fig2, ax2 = plt.subplots(figsize=(8, 3.5))
ax2.plot(iters[found], residue[found], "o-", ms=3, lw=1, color="tab:green")
ax2.axhspan(0.0, 1.0, color="tab:green", alpha=0.08, label="elliptic band (O-point)")
ax2.set(xlabel="Optimisation iterate", ylabel="Greene residue R", title="Island O-point stability")
ax2.legend()
fig2.tight_layout()
plt.show()

print(f"Final Greene residue: {residue[found][-1]:.3e}  (elliptic O-point)")
print(f"Final iota: {iota[found][-1]:.6f}  (3/5 = {3 / 5:.6f}, rel err {iota_rel[found][-1]:.1e})")

Final Greene residue: 1.892e-01  (elliptic O-point)
Final iota: 0.600000  (3/5 = 0.600000, rel err 1.8e-07)
